# Part 10: Validation Methods for Time Series

---------------------------------------

This is part of a series of notebooks about practical time series methods:

* [Part 0: the basics](https://www.kaggle.com/konradb/ts-0-the-basics)
* [Part 1a: smoothing methods](https://www.kaggle.com/konradb/ts-1a-smoothing-methods)
* [Part 1b: Prophet](https://www.kaggle.com/konradb/ts-1b-prophet) 
* [Part 2: ARMA](https://www.kaggle.com/konradb/ts-2-arma-and-friends)
* [Part 3: Time series for finance](https://www.kaggle.com/konradb/ts-3-time-series-for-finance) 
* [Part 4: Sales and demand forecasting](https://www.kaggle.com/konradb/ts-4-sales-and-demand-forecasting)
* [Part 5: Automatic for the people](https://www.kaggle.com/code/konradb/ts-5-automatic-for-the-people) 
* [Part 6: Deep learning for TS - sequences](https://www.kaggle.com/konradb/ts-6-deep-learning-for-ts-sequences)
* [Part 7: Survival analysis](https://www.kaggle.com/konradb/ts-7-survival-analysis)
* [Part 8: Hierarchical time series](https://www.kaggle.com/konradb/ts-8-hierarchical-time-series) 
* [Part 9: Hybrid methods](https://www.kaggle.com/konradb/ts-9-hybrid-methods) 
* **Part 10: Validation methods for time series** - this notebook
* [Part 11: Transfer learning](https://www.kaggle.com/code/konradb/ts-11-deep-learning-for-ts-transfer-learning)

---------------------------------------

With the exception of people who deploy to production on a Friday evening because YOLO, we all agree that model validation matters: measuring the performance of an ML model (and hence its generalisation power) allows us to assess the robustness, optimise parameters and estimate performance on unseen data. If there is a good reason to believe the underlying data generating process is stationary (no concept drift), you are usually fine with training-validation-test split (overfitting to validation set notwithstanding). It becomes slightly more complicated if the temporal dimension matters: in this episode we will walk through different manners of assessing performance of time series models without breaking the arrow of time.

## Sections

* [Setup & Data](#section-setup)
* [Random split](#section-one)
* [KFold](#section-two)
* [Walk forward](#section-three)
* [Group Time Series](#section-four)
* [Purged Group KFold](#section-five)
* [Combinatorial Purged Group KFold](#section-six)
* [Results Summary](#section-summary)

<a id="section-setup"></a>
# Setup & Data

In [6]:
from __future__ import annotations

import logging
from dataclasses import dataclass, field

import lightgbm as lgb
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import yfinance as yf
from sklearn.model_selection import (
    GroupKFold,
    KFold,
    TimeSeriesSplit,
    train_test_split,
)
from sklearn.model_selection._split import _BaseKFold, _num_samples, indexable

logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s | %(levelname)s | %(message)s",
    datefmt="%H:%M:%S",
)
log = logging.getLogger(__name__)

In [7]:
@dataclass
class Config:
    """Central configuration for the notebook."""

    seed: int = 42
    n_folds: int = 5
    target: str = "target"
    val_size: float = 0.33
    fig_size: tuple[int, int] = (20, 6)

    tickers: list[str] = field(
        default_factory=lambda: ["AAPL", "MSFT", "GOOG", "AMZN", "META"]
    )
    start_date: str = "2015-01-01"
    end_date: str = "2024-12-31"
    forecast_horizon: int = 5  # days

    lgb_params: dict = field(
        default_factory=lambda: {
            "n_estimators": 200,
            "learning_rate": 0.05,
            "max_depth": 6,
            "num_leaves": 31,
            "subsample": 0.8,
            "colsample_bytree": 0.8,
            "random_state": 42,
            "verbosity": -1,
        }
    )


CFG = Config()
np.random.seed(CFG.seed)
plt.rcParams.update({"figure.figsize": CFG.fig_size})

## Data: Multi-stock Financial Dataset via `yfinance`

We use a portfolio of well-known tech stocks. This gives us:
- A natural **time dimension** (trading dates)
- A natural **group dimension** (ticker / investment)
- A realistic setting for exploring purged and group-aware CV schemes

The target is the **5-day forward return**, which makes the purging discussion especially relevant: labels depend on future price paths.

In [8]:
def load_stock_data(cfg: Config) -> pd.DataFrame:
    """Download OHLCV data for multiple tickers and stack into a long DataFrame."""
    frames = []
    for ticker in cfg.tickers:
        df = yf.download(ticker, start=cfg.start_date, end=cfg.end_date, progress=False)
        df = df.droplevel("Ticker", axis=1) if isinstance(df.columns, pd.MultiIndex) else df
        df = df.reset_index()
        df["ticker"] = ticker
        frames.append(df)
    return pd.concat(frames, ignore_index=True)


raw = load_stock_data(CFG)
log.info(f"Raw data shape: {raw.shape}")
raw.head()

00:56:14 | INFO | Raw data shape: (12575, 7)


Price,Date,Close,High,Low,Open,Volume,ticker
0,2015-01-02,24.214899,24.682231,23.776359,24.671157,212818400,AAPL
1,2015-01-05,23.532722,24.064285,23.346676,23.984551,257142000,AAPL
2,2015-01-06,23.534931,23.794068,23.173911,23.596947,263188400,AAPL
3,2015-01-07,23.864954,23.964621,23.632395,23.743137,160423600,AAPL
4,2015-01-08,24.781900,24.839487,24.075364,24.192753,237458000,AAPL


In [9]:
def engineer_features(df: pd.DataFrame, cfg: Config) -> pd.DataFrame:
    """Create lag, rolling, and return features per ticker."""
    df = df.sort_values(["ticker", "Date"]).reset_index(drop=True)

    # Assign a time_id (integer) to each unique trading date
    date_map = {d: i for i, d in enumerate(sorted(df["Date"].unique()))}
    df["time_id"] = df["Date"].map(date_map)

    out = []
    for _, grp in df.groupby("ticker"):
        g = grp.copy()

        # Forward return as target (percentage)
        g[cfg.target] = g["Close"].pct_change(cfg.forecast_horizon).shift(-cfg.forecast_horizon) * 100

        # Lag features
        for lag in [1, 2, 3, 5, 10, 20]:
            g[f"ret_{lag}d"] = g["Close"].pct_change(lag) * 100

        # Rolling statistics
        for win in [5, 10, 20, 60]:
            g[f"vol_{win}d"] = g["Close"].pct_change().rolling(win).std() * 100
            g[f"sma_ratio_{win}d"] = g["Close"] / g["Close"].rolling(win).mean()

        # Volume features
        g["vol_change_5d"] = g["Volume"].pct_change(5)
        g["vol_sma_ratio_20d"] = g["Volume"] / g["Volume"].rolling(20).mean()

        # Range feature
        g["high_low_range"] = (g["High"] - g["Low"]) / g["Close"] * 100

        out.append(g)

    result = pd.concat(out, ignore_index=True).dropna()
    return result


data = engineer_features(raw, CFG)
log.info(f"Feature-engineered data shape: {data.shape}")
data.head()

00:56:14 | INFO | Feature-engineered data shape: (12250, 26)


Price,Date,Close,High,Low,Open,Volume,ticker,time_id,target,ret_1d,...,sma_ratio_5d,vol_10d,sma_ratio_10d,vol_20d,sma_ratio_20d,vol_60d,sma_ratio_60d,vol_change_5d,vol_sma_ratio_20d,high_low_range
60,2015-03-31,27.668070,28.126128,27.652505,28.037184,168362400,AAPL,60,0.940284,-1.535168,...,1.000772,1.527011,0.989550,1.440111,0.990326,1.753639,1.029993,0.281597,0.799647,1.711804
61,2015-04-01,27.628046,27.821499,27.372333,27.754790,162485600,AAPL,61,1.859165,-0.144657,...,0.997928,1.455953,0.991446,1.436221,0.990584,1.708086,1.025896,-0.213605,0.765226,1.625758
62,2015-04-02,27.865969,27.919334,27.614704,27.801485,128880400,AAPL,62,1.420369,0.861164,...,1.004779,1.492034,1.001726,1.408159,0.999549,1.709321,1.031965,-0.322722,0.621178,1.093198
63,2015-04-06,28.317366,28.352944,27.645843,27.676974,148776000,AAPL,63,-0.392626,1.619886,...,1.014386,1.534473,1.016775,1.455811,1.015437,1.711937,1.045808,-0.059480,0.742589,2.497056
64,2015-04-07,28.019396,28.488571,28.012725,28.381840,140049200,AAPL,64,0.230153,-1.052251,...,1.004288,1.538473,1.007041,1.472582,1.005205,1.655976,1.032745,-0.256634,0.738482,1.698271


In [10]:
def prepare_features(df: pd.DataFrame, cfg: Config) -> tuple[pd.DataFrame, pd.Series, list[str]]:
    """Separate features, target, and return the feature list."""
    drop_cols = ["Date", "ticker", "time_id", cfg.target, "Open", "High", "Low", "Close", "Volume"]
    features = [c for c in df.columns if c not in drop_cols]
    X = df[features].copy()
    y = df[cfg.target].copy()
    return X, y, features


def reduce_memory(df: pd.DataFrame) -> pd.DataFrame:
    """Downcast numeric columns to reduce memory footprint."""
    for col in df.select_dtypes(include=["int", "float"]).columns:
        df[col] = pd.to_numeric(df[col], downcast="float")
    return df


data = reduce_memory(data)

# Hold-out test set: last 20% of time_ids
cutoff_time = int(data["time_id"].max() * 0.8)
train_data = data[data["time_id"] <= cutoff_time].copy().reset_index(drop=True)
test_data = data[data["time_id"] > cutoff_time].copy().reset_index(drop=True)

X_train_full, y_train_full, feature_cols = prepare_features(train_data, CFG)
X_test, y_test, _ = prepare_features(test_data, CFG)

log.info(f"Train shape: {train_data.shape}, Test shape: {test_data.shape}")
log.info(f"Features ({len(feature_cols)}): {feature_cols}")

00:56:14 | INFO | Train shape: (9740, 26), Test shape: (2510, 26)
00:56:14 | INFO | Features (17): ['ret_1d', 'ret_2d', 'ret_3d', 'ret_5d', 'ret_10d', 'ret_20d', 'vol_5d', 'sma_ratio_5d', 'vol_10d', 'sma_ratio_10d', 'vol_20d', 'sma_ratio_20d', 'vol_60d', 'sma_ratio_60d', 'vol_change_5d', 'vol_sma_ratio_20d', 'high_low_range']


## Helper Functions

Central `train_and_evaluate` and `run_cv_experiment` to avoid repeating training logic across every validation section.

In [11]:
def train_and_evaluate(
    X_train: pd.DataFrame,
    y_train: pd.Series,
    X_val: pd.DataFrame,
    y_val: pd.Series,
    params: dict,
) -> tuple[float, float, lgb.LGBMRegressor]:
    """Train a LightGBM model and return (val_rmse, val_corr, model)."""
    model = lgb.LGBMRegressor(**params)
    model.fit(
        X_train,
        y_train,
        eval_set=[(X_val, y_val)],
        eval_metric="rmse",
        callbacks=[
            lgb.early_stopping(stopping_rounds=30, verbose=False),
            lgb.log_evaluation(period=0),
        ],
    )
    preds = model.predict(X_val)
    val_corr = pd.Series(preds).corr(pd.Series(y_val.values))
    val_rmse = np.sqrt(np.mean((preds - y_val.values) ** 2))
    return val_rmse, val_corr, model


def run_cv_experiment(
    df: pd.DataFrame,
    cv_splitter,
    features: list[str],
    cfg: Config,
    X_holdout: pd.DataFrame | None = None,
    y_holdout: pd.Series | None = None,
    groups: pd.Series | None = None,
    method_name: str = "CV",
) -> dict:
    """Run a full cross-validation experiment and return a results dict."""
    X_all = df[features]
    y_all = df[cfg.target]

    split_args = {"X": X_all}
    if groups is not None:
        split_args["groups"] = groups

    val_scores, test_scores = [], []

    for fold, (trn_idx, val_idx) in enumerate(cv_splitter.split(**split_args)):
        X_tr, y_tr = X_all.iloc[trn_idx], y_all.iloc[trn_idx]
        X_vl, y_vl = X_all.iloc[val_idx], y_all.iloc[val_idx]

        val_rmse, val_corr, model = train_and_evaluate(X_tr, y_tr, X_vl, y_vl, cfg.lgb_params)
        val_scores.append(val_corr)

        test_corr = None
        if X_holdout is not None and y_holdout is not None:
            test_preds = model.predict(X_holdout)
            test_corr = pd.Series(test_preds).corr(pd.Series(y_holdout.values))
            test_scores.append(test_corr)

        log.info(
            f"[{method_name}] Fold {fold + 1}: "
            f"val_corr={val_corr:.4f}, val_rmse={val_rmse:.4f}"
            + (f", test_corr={test_corr:.4f}" if test_corr is not None else "")
        )

    mean_val = np.mean(val_scores)
    std_val = np.std(val_scores)
    mean_test = np.mean(test_scores) if test_scores else None
    log.info(f"[{method_name}] Mean val_corr: {mean_val:.4f} +/- {std_val:.4f}")
    if mean_test is not None:
        log.info(f"[{method_name}] Mean test_corr: {mean_test:.4f}")

    return {
        "method": method_name,
        "val_scores": val_scores,
        "test_scores": test_scores,
        "mean_val_corr": mean_val,
        "std_val_corr": std_val,
        "mean_test_corr": mean_test,
    }

In [12]:
def plot_cv_splits(
    cv_splitter,
    X: pd.DataFrame,
    title: str = "CV Splits",
    groups: pd.Series | None = None,
) -> None:
    """Visualize train/validation index assignments for each fold."""
    n = len(X)
    split_args = {"X": X}
    if groups is not None:
        split_args["groups"] = groups

    splits = list(cv_splitter.split(**split_args))
    n_splits = len(splits)

    fig, ax = plt.subplots(figsize=(CFG.fig_size[0], max(2, n_splits * 0.6)))

    for i, (train_idx, val_idx) in enumerate(splits):
        indices = np.full(n, np.nan)
        indices[train_idx] = 0
        indices[val_idx] = 1
        ax.scatter(
            range(n), [i + 0.5] * n,
            c=indices, cmap=plt.cm.coolwarm, marker="|", s=2, vmin=-0.2, vmax=1.2,
        )

    ax.set_yticks(np.arange(n_splits) + 0.5)
    ax.set_yticklabels([f"Fold {i + 1}" for i in range(n_splits)])
    ax.set_xlabel("Sample index")
    ax.set_title(title + "  (blue = train, red = validation)")
    ax.invert_yaxis()
    plt.tight_layout()
    plt.show()

---
<a id="section-one"></a>
# 1. Random Split (`train_test_split`)

The simplest approach: randomly split into train and validation. This **ignores** the temporal order entirely, which in a time series context leads to **data leakage** — the model sees future data during training.

In [14]:
X_tr, X_vl, y_tr, y_vl = train_test_split(
    X_train_full, y_train_full, test_size=CFG.val_size, random_state=CFG.seed
)

val_rmse, val_corr, model_random = train_and_evaluate(X_tr, y_tr, X_vl, y_vl, CFG.lgb_params)

test_preds = model_random.predict(X_test)
test_corr = pd.Series(test_preds).corr(pd.Series(y_test.values))

log.info(f"[RandomSplit] val_corr={val_corr:.4f}, val_rmse={val_rmse:.4f}")
log.info(f"[RandomSplit] test_corr={test_corr:.4f}")

results_random = {
    "method": "RandomSplit",
    "val_scores": [val_corr],
    "test_scores": [test_corr],
    "mean_val_corr": val_corr,
    "std_val_corr": 0.0,
    "mean_test_corr": test_corr,
}

LightGBMError: scikit-learn is required for lightgbm.sklearn. You must install scikit-learn and restart your session to use this module.

---
<a id="section-two"></a>
# 2. KFold

Standard K-Fold cross-validation shuffles data randomly across folds. Like random split, it **breaks the arrow of time** — future observations leak into training folds.

In [ ]:
kf = KFold(n_splits=CFG.n_folds, shuffle=True, random_state=CFG.seed)

plot_cv_splits(kf, X_train_full, title="KFold")

results_kfold = run_cv_experiment(
    train_data, kf, feature_cols, CFG,
    X_holdout=X_test, y_holdout=y_test,
    method_name="KFold",
)

There is typically substantial overfit occurring — validation scores look better than test scores because the model has seen 'future' data during training.

---
<a id="section-three"></a>
# 3. Walk Forward (TimeSeriesSplit)

The scikit-learn implementation of a time series split (a.k.a walk forward validation) is a variation of `KFold`:

* In the $k$th split, it returns first $k$ folds as train set and the $(k+1)$th fold as test set
* Successive training sets can be supersets of those that came before them or have a fixed size $\rightarrow$ controlled by `max_train_size`
* Useful in time-sensitive contexts like trading $\rightarrow$ robustness of a strategy, concept drift
* The `gap` parameter prevents leakage when features include rolling windows

This **respects the arrow of time**: training always comes before validation.

In [ ]:
tscv = TimeSeriesSplit(n_splits=CFG.n_folds, gap=20)

plot_cv_splits(tscv, X_train_full, title="TimeSeriesSplit (Walk Forward)")

results_walkfwd = run_cv_experiment(
    train_data, tscv, feature_cols, CFG,
    X_holdout=X_test, y_holdout=y_test,
    method_name="WalkForward",
)

---
<a id="section-four"></a>
# 4. Group Time Series Split

Standard `GroupKFold` ensures that the same group (e.g. the same trading day across all stocks) doesn't appear in both train and validation. However, it doesn't respect temporal order.

**GroupTimeSeriesSplit** combines the best of both worlds:
- Groups are kept intact (no leakage across groups)
- Temporal order is respected (train < validation in time)

Credit: [Gaurav Chawla](https://github.com/getgaurav2/) for the original implementation.

In [ ]:
class GroupTimeSeriesSplit(_BaseKFold):
    """Time Series CV with non-overlapping groups that respects temporal order.

    In each split, test groups are strictly after train groups.
    The same group never appears in both train and test.

    Based on: https://github.com/getgaurav2/scikit-learn
    """

    def __init__(self, n_splits: int = 5, *, max_train_size: int | None = None):
        super().__init__(n_splits, shuffle=False, random_state=None)
        self.max_train_size = max_train_size

    def split(self, X, y=None, groups=None):
        if groups is None:
            raise ValueError("The 'groups' parameter should not be None")
        X, y, groups = indexable(X, y, groups)
        n_samples = _num_samples(X)
        n_splits = self.n_splits
        n_folds = n_splits + 1

        group_dict: dict[int, list[int]] = {}
        u, ind = np.unique(groups, return_index=True)
        unique_groups = u[np.argsort(ind)]
        n_groups = _num_samples(unique_groups)

        for idx in range(n_samples):
            group_dict.setdefault(groups[idx], []).append(idx)

        if n_folds > n_groups:
            raise ValueError(
                f"Cannot have number of folds={n_folds} greater than "
                f"the number of groups={n_groups}"
            )

        group_test_size = n_groups // n_folds
        group_test_starts = range(
            n_groups - n_splits * group_test_size, n_groups, group_test_size
        )

        for group_test_start in group_test_starts:
            train_array: np.ndarray = np.array([], dtype=int)
            test_array: np.ndarray = np.array([], dtype=int)

            for train_group_idx in unique_groups[:group_test_start]:
                train_array = np.sort(
                    np.unique(np.concatenate((train_array, group_dict[train_group_idx])))
                )

            if self.max_train_size and self.max_train_size < len(train_array):
                train_array = train_array[-self.max_train_size :]

            for test_group_idx in unique_groups[
                group_test_start : group_test_start + group_test_size
            ]:
                test_array = np.sort(
                    np.unique(np.concatenate((test_array, group_dict[test_group_idx])))
                )

            yield [int(i) for i in train_array], [int(i) for i in test_array]

In [ ]:
groups_train = train_data["time_id"]
gtscv = GroupTimeSeriesSplit(n_splits=CFG.n_folds)

plot_cv_splits(gtscv, X_train_full, title="GroupTimeSeriesSplit", groups=groups_train)

results_group_ts = run_cv_experiment(
    train_data, gtscv, feature_cols, CFG,
    X_holdout=X_test, y_holdout=y_test,
    groups=groups_train,
    method_name="GroupTimeSeriesSplit",
)

---
<a id="section-five"></a>
# 5. Purged Group Time Series Split

In financial ML, labels are **path-dependent**: a data point's label depends on future price movements over a window. This means adjacent time periods have overlapping information.

**Purging** removes data points from the training set whose label windows overlap with the test period's trade times, preventing information leakage.

Key parameters:
- `group_gap`: number of groups to skip between train and test (the purge zone)
- `max_train_group_size` / `max_test_group_size`: control fold sizes

Based on: [marketneutral's implementation](https://www.kaggle.com/code/marketneutral/purged-time-series-cv-xgboost-optuna)

In [ ]:
class PurgedGroupTimeSeriesSplit(_BaseKFold):
    """Time Series CV with group-aware purging.

    Adds a gap between train and test to prevent leakage from
    windowed or lag features.
    """

    def __init__(
        self,
        n_splits: int = 5,
        *,
        max_train_group_size: int | float = np.inf,
        max_test_group_size: int | float = np.inf,
        group_gap: int = 0,
    ):
        super().__init__(n_splits, shuffle=False, random_state=None)
        self.max_train_group_size = max_train_group_size
        self.group_gap = group_gap
        self.max_test_group_size = max_test_group_size

    def split(self, X, y=None, groups=None):
        if groups is None:
            raise ValueError("The 'groups' parameter should not be None")
        X, y, groups = indexable(X, y, groups)
        n_samples = _num_samples(X)
        n_splits = self.n_splits
        group_gap = self.group_gap
        max_test_group_size = self.max_test_group_size
        max_train_group_size = self.max_train_group_size
        n_folds = n_splits + 1

        group_dict: dict[int, list[int]] = {}
        u, ind = np.unique(groups, return_index=True)
        unique_groups = u[np.argsort(ind)]
        n_groups = _num_samples(unique_groups)

        for idx in range(n_samples):
            group_dict.setdefault(groups[idx], []).append(idx)

        if n_folds > n_groups:
            raise ValueError(
                f"Cannot have number of folds={n_folds} greater than "
                f"the number of groups={n_groups}"
            )

        group_test_size = min(n_groups // n_folds, max_test_group_size)
        group_test_starts = range(
            n_groups - n_splits * group_test_size, n_groups, group_test_size
        )

        for group_test_start in group_test_starts:
            train_array: np.ndarray = np.array([], dtype=int)
            test_array: np.ndarray = np.array([], dtype=int)

            group_st = max(0, group_test_start - group_gap - max_train_group_size)
            for train_group_idx in unique_groups[group_st : group_test_start - group_gap]:
                train_array = np.sort(
                    np.unique(np.concatenate((train_array, group_dict[train_group_idx])))
                )

            for test_group_idx in unique_groups[
                group_test_start : group_test_start + group_test_size
            ]:
                test_array = np.sort(
                    np.unique(np.concatenate((test_array, group_dict[test_group_idx])))
                )

            test_array = test_array[group_gap:]

            yield [int(i) for i in train_array], [int(i) for i in test_array]

In [ ]:
purged_cv = PurgedGroupTimeSeriesSplit(
    n_splits=CFG.n_folds,
    group_gap=10,
    max_train_group_size=500,
    max_test_group_size=100,
)

plot_cv_splits(purged_cv, X_train_full, title="PurgedGroupTimeSeriesSplit", groups=groups_train)

results_purged = run_cv_experiment(
    train_data, purged_cv, feature_cols, CFG,
    X_holdout=X_test, y_holdout=y_test,
    groups=groups_train,
    method_name="PurgedGroupTS",
)

---
<a id="section-six"></a>
# 6. Combinatorial Purged Group KFold

The combinatorial variant (from Marcos López de Prado's *Advances in Financial Machine Learning*) generates **all possible combinations** of test groups from the available folds, providing many more unique train/test splits than standard sequential methods.

This is the most sophisticated validation scheme, combining:
- Group awareness
- Purging (gap between train and test)
- Combinatorial path generation for more robust performance estimates

In [ ]:
from itertools import combinations


class CombinatorialPurgedGroupKFold(_BaseKFold):
    """Combinatorial Purged Group KFold.

    Generates all C(n_splits, n_test_splits) combinations of test groups,
    with purging between adjacent train and test groups.
    """

    def __init__(self, n_splits: int = 5, n_test_splits: int = 1, *, group_gap: int = 0):
        super().__init__(n_splits, shuffle=False, random_state=None)
        self.n_test_splits = n_test_splits
        self.group_gap = group_gap

    def split(self, X, y=None, groups=None):
        if groups is None:
            raise ValueError("The 'groups' parameter should not be None")
        X, y, groups = indexable(X, y, groups)
        n_samples = _num_samples(X)

        u, ind = np.unique(groups, return_index=True)
        unique_groups = u[np.argsort(ind)]
        n_groups = len(unique_groups)

        group_dict: dict = {}
        for idx in range(n_samples):
            group_dict.setdefault(groups[idx], []).append(idx)

        if self.n_splits > n_groups:
            raise ValueError(
                f"Cannot have n_splits={self.n_splits} > n_groups={n_groups}"
            )

        group_test_size = n_groups // self.n_splits
        group_bins = []
        for i in range(self.n_splits):
            start = i * group_test_size
            end = start + group_test_size if i < self.n_splits - 1 else n_groups
            group_bins.append(list(range(start, end)))

        for test_bin_combo in combinations(range(self.n_splits), self.n_test_splits):
            test_group_indices = []
            for bin_idx in test_bin_combo:
                test_group_indices.extend(group_bins[bin_idx])
            test_group_set = set(test_group_indices)

            purge_set = set()
            for tgi in test_group_indices:
                for gap in range(1, self.group_gap + 1):
                    purge_set.add(tgi - gap)
                    purge_set.add(tgi + gap)

            train_group_indices = [
                i for i in range(n_groups)
                if i not in test_group_set and i not in purge_set
            ]

            train_indices = []
            for gi in train_group_indices:
                train_indices.extend(group_dict[unique_groups[gi]])

            test_indices = []
            for gi in test_group_indices:
                test_indices.extend(group_dict[unique_groups[gi]])

            yield sorted(train_indices), sorted(test_indices)

In [ ]:
cpcv = CombinatorialPurgedGroupKFold(n_splits=5, n_test_splits=1, group_gap=2)

plot_cv_splits(cpcv, X_train_full, title="CombinatorialPurgedGroupKFold", groups=groups_train)

results_cpcv = run_cv_experiment(
    train_data, cpcv, feature_cols, CFG,
    X_holdout=X_test, y_holdout=y_test,
    groups=groups_train,
    method_name="CPCV",
)

---
<a id="section-summary"></a>
# Results Summary

Let's compare all validation methods side-by-side.

In [ ]:
all_results = [
    results_random,
    results_kfold,
    results_walkfwd,
    results_group_ts,
    results_purged,
    results_cpcv,
]

summary_df = pd.DataFrame(
    [
        {
            "Method": r["method"],
            "Mean Val Corr": r["mean_val_corr"],
            "Std Val Corr": r["std_val_corr"],
            "Mean Test Corr": r["mean_test_corr"],
            "Overfit (Val - Test)": (
                r["mean_val_corr"] - r["mean_test_corr"]
                if r["mean_test_corr"] is not None
                else None
            ),
        }
        for r in all_results
    ]
)

summary_df.style.format(
    {
        "Mean Val Corr": "{:.4f}",
        "Std Val Corr": "{:.4f}",
        "Mean Test Corr": "{:.4f}",
        "Overfit (Val - Test)": "{:.4f}",
    }
).background_gradient(subset=["Overfit (Val - Test)"], cmap="RdYlGn_r")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 5))

methods = summary_df["Method"]
x_pos = np.arange(len(methods))

axes[0].bar(x_pos, summary_df["Mean Val Corr"], alpha=0.7, label="Val Corr")
axes[0].bar(x_pos, summary_df["Mean Test Corr"], alpha=0.7, label="Test Corr")
axes[0].set_xticks(x_pos)
axes[0].set_xticklabels(methods, rotation=30, ha="right")
axes[0].set_ylabel("Correlation")
axes[0].set_title("Validation vs Test Correlation by Method")
axes[0].legend()
axes[0].axhline(y=0, color="gray", linestyle="--", alpha=0.5)

overfit = summary_df["Overfit (Val - Test)"].fillna(0)
colors = ["green" if v <= 0.01 else "orange" if v <= 0.05 else "red" for v in overfit]
axes[1].bar(x_pos, overfit, color=colors, alpha=0.7)
axes[1].set_xticks(x_pos)
axes[1].set_xticklabels(methods, rotation=30, ha="right")
axes[1].set_ylabel("Overfit Gap")
axes[1].set_title("Overfitting: Val Corr - Test Corr (lower is better)")
axes[1].axhline(y=0, color="gray", linestyle="--", alpha=0.5)

plt.tight_layout()
plt.show()

## Key Takeaways

1. **Random split and KFold** tend to overestimate model performance by leaking future information into training.

2. **Walk Forward (TimeSeriesSplit)** respects temporal order but doesn't handle group structure (e.g. multiple assets on the same date).

3. **GroupTimeSeriesSplit** handles both temporal order and group structure — suitable for most time series applications.

4. **PurgedGroupTimeSeriesSplit** adds a gap to prevent leakage from windowed/lag features — essential for financial ML.

5. **Combinatorial Purged Group KFold** generates more path combinations for more robust estimates — the gold standard for financial backtesting.

The gap between validation and test scores is a proxy for overfitting. Time-aware methods generally produce validation estimates closer to true out-of-sample performance.

---

### Recommended Reading

- López de Prado, M. (2018). *Advances in Financial Machine Learning*. Wiley.
- [Cross-Validation, Embargo, Purging & Combinatorial](https://blog.quantinsti.com/cross-validation-embargo-purging-combinatorial/)
- [Walk-Forward Optimization](https://audhiaprilliant.medium.com/walk-forward-optimization-cross-validation-technique-for-time-series-data-61739f58f2c0)
- [A Survey of Cross-Validation Procedures for Model Selection (arXiv:2203.10716)](https://arxiv.org/abs/2203.10716)